In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries. """

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [16]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemma4",
    model_provider="openai",
    api_key="dummy",
    base_url="http://localhost:8080/v1"
)

agent = create_agent(
    model=model,
    system_prompt="Plan first then act.",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [17]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

There are 275 artists in the database.


In [30]:
from langchain_core.messages import HumanMessage

for step in agent.stream(
    {"messages": [HumanMessage(content="How many artists are in the database with more than 5 tracks?")]},
    context={"user_role": "internal"},
    stream_mode="updates"
):
    for node_name, node_output in step.items():
        # Skip the tools node to hide raw database output
        if node_name == "tools":
            continue

        if "messages" in node_output:
            for msg in node_output["messages"]:
                # Print tool invocation details
                if getattr(msg, "tool_calls", None):
                    for tc in msg.tool_calls:
                        print(f"[Tool Call] {tc['name']} -> {tc['args']}")
                
                # Print the final AI text
                if msg.content:
                    print(f"\n[AI]: {msg.content}")

[Tool Call] sql_query -> {'query': 'PRAGMA_LIST'}
[Tool Call] sql_query -> {'query': 'SELECT COUNT(artist_id) FROM (SELECT artist_id FROM Tracks GROUP BY artist_id HAVING COUNT(id) > 5);'}
[Tool Call] sql_query -> {'query': "SELECT name FROM sqlite_master WHERE type='table';"}
[Tool Call] sql_query -> {'query': 'PRAGMA table_info(Artist);'}
[Tool Call] sql_query -> {'query': 'PRAGMA table_info(Track);'}
[Tool Call] sql_query -> {'query': 'PRAGMA table_info(Album);'}
[Tool Call] sql_query -> {'query': 'SELECT COUNT(T1.ArtistId) FROM Album AS T1 INNER JOIN Track AS T2 ON T1.AlbumId = T2.AlbumId GROUP BY T1.ArtistId HAVING COUNT(T2.TrackId) > 5;'}
[Tool Call] sql_query -> {'query': 'SELECT COUNT(artist_id) FROM (SELECT T1.ArtistId FROM Album AS T1 INNER JOIN Track AS T2 ON T1.AlbumId = T2.AlbumId GROUP BY T1.ArtistId HAVING COUNT(T2.TrackId) > 5) AS ArtistCounts;'}
[Tool Call] sql_query -> {'query': 'SELECT COUNT(ArtistId) FROM (SELECT T1.ArtistId FROM Album AS T1 INNER JOIN Track AS T2 O